In [ ]:
from utils_monoBP_single import *
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
import matplotlib.pyplot as plt
import math
import pandas as pd
import time
import sys
from scipy.stats import ks_2samp
import ot
from torch.distributions import MultivariateNormal

In [ ]:
def normalized_weight(log_w):
    tmp_w = (log_w - log_w.max()).exp()
    tmp_w = tmp_w / tmp_w.sum()

    thresh_id = 0
    while tmp_w.max() > 100 / log_w.shape[0]:
        thresh_id += 1
        clip = log_w.sort(descending=True).values[thresh_id]
        log_w = torch.clamp(log_w, max = clip)
        tmp_w = (log_w - log_w.max()).exp()
        tmp_w = tmp_w / tmp_w.sum()
    return tmp_w

def weighted_quantile(array, weight, quantile):
    # weight sums to 1
    sorter = np.argsort(array)
    sorted_array = array[sorter]
    sorted_weight = weight[sorter]
    index = np.searchsorted(np.cumsum(sorted_weight), quantile, side='left')
    return sorted_array[index]

def weighted_ks_2samp(x, y, wx=None, wy=None):
    x = np.asarray(x)
    y = np.asarray(y)

    if wx is None:
        wx = np.ones(len(x)) / len(x)
    else:
        wx = np.asarray(wx, dtype=float)
        wx = wx / wx.sum()

    if wy is None:
        wy = np.ones(len(y)) / len(y)
    else:
        wy = np.asarray(wy, dtype=float)
        wy = wy / wy.sum()

    # sort samples
    ix = np.argsort(x)
    iy = np.argsort(y)

    x_sorted = x[ix]
    y_sorted = y[iy]
    wx_sorted = wx[ix]
    wy_sorted = wy[iy]

    # all unique evaluation points
    grid = np.sort(np.unique(np.concatenate([x_sorted, y_sorted])))

    # weighted CDF values at grid
    cdf_x = np.searchsorted(x_sorted, grid, side="right")
    cdf_y = np.searchsorted(y_sorted, grid, side="right")

    cdf_x_vals = np.concatenate([[0.0], np.cumsum(wx_sorted)])[cdf_x]
    cdf_y_vals = np.concatenate([[0.0], np.cumsum(wy_sorted)])[cdf_y]

    return np.max(np.abs(cdf_x_vals - cdf_y_vals))


# Averaged Results

In [ ]:
task_set = [i for i in range(10)]

In [ ]:
KS_fisherDR_cross = torch.zeros(len(task_set), 101)
KS_nmodel_fisher = torch.zeros(len(task_set), 101)
KS_nmodel_fisher_5x = torch.zeros(len(task_set), 101)
KS_BSL = torch.zeros(len(task_set), 101)
KS_ABCW1 = torch.zeros(len(task_set), 101)
KS_NPE = torch.zeros(len(task_set), 101)
KS_NLE = torch.zeros(len(task_set), 101)



Cvg_fisherDR_cross = torch.zeros(len(task_set), 101)
Cvg_nmodel_fisher = torch.zeros(len(task_set), 101)
Cvg_nmodel_fisher_5x = torch.zeros(len(task_set), 101)
Cvg_BSL = torch.zeros(len(task_set), 101)
Cvg_ABCW1 = torch.zeros(len(task_set), 101)
Cvg_NPE = torch.zeros(len(task_set), 101)
Cvg_gibbs = torch.zeros(len(task_set), 101)
Cvg_NLE = torch.zeros(len(task_set), 101)


Width_fisherDR_cross = torch.zeros(len(task_set), 101)
Width_nmodel_fisher = torch.zeros(len(task_set), 101)
Width_nmodel_fisher_5x = torch.zeros(len(task_set), 101)
Width_BSL = torch.zeros(len(task_set), 101)
Width_ABCW1 = torch.zeros(len(task_set), 101)
Width_NPE = torch.zeros(len(task_set), 101)
Width_gibbs = torch.zeros(len(task_set), 101)
Width_NLE = torch.zeros(len(task_set), 101)


W1_fisherDR_cross = torch.zeros(len(task_set), 101)
W1_nmodel_fisher = torch.zeros(len(task_set), 101)
W1_nmodel_fisher_5x = torch.zeros(len(task_set), 101)
W1_BSL = torch.zeros(len(task_set), 101)
W1_ABCW1 = torch.zeros(len(task_set), 101)
W1_NPE = torch.zeros(len(task_set), 101)
W1_NLE = torch.zeros(len(task_set), 101)

In [ ]:
for j in tqdm(range(len(task_set))):
    task_id = task_set[j]
    pred_ys_BSL = torch.tensor( pd.read_csv(f'sample_res_others/pred_ys_BSL_task{task_id}.csv', header=None).values, 
                         dtype = torch.float32).contiguous()
    pred_ys_ABCW1 = torch.tensor( pd.read_csv(f'sample_res_others/pred_ys_ABCW1_task{task_id}.csv', header=None).values, 
                             dtype = torch.float32).contiguous()
    
    pred_ys_gibbs = torch.tensor( pd.read_csv(f'sample_res/pred_ys_gibbs_task{task_id}.csv', header=None).values, 
                                 dtype = torch.float32).contiguous()
    pred_ys_fisherDR_cross = torch.tensor( pd.read_csv(f'sample_res_DebReg_fisher_cross/pred_ys_ann_task{task_id}.csv', header=None).values, 
                                 dtype = torch.float32).contiguous()
    pred_ys_nmodel_fisher = torch.tensor( pd.read_csv(f'sample_res_nmodel_fisher/pred_ys_ann_task{task_id}_trainsize100000.csv', header=None).values, 
                                 dtype = torch.float32).contiguous()
    pred_ys_nmodel_fisher_5x = torch.tensor( pd.read_csv(f'sample_res_nmodel_fisher/pred_ys_ann_task{task_id}_trainsize500000.csv', header=None).values, 
                             dtype = torch.float32).contiguous()


    theta_r1_ABCW1 = torch.tensor(np.load(f"sample_res_others/theta_r1_ABCW1_task{task_id}.npy"), dtype=torch.float32)


    # NLE res
    theta_r1_NLE = torch.tensor(np.load(f"res_NLE/theta_post{task_id}.npy"), dtype=torch.float32)
    A = get_A(M)
    grids = torch.arange(0, 1.01, 0.01)
    psi_grids = get_psi(grids, M)

    if task_id == 6: # some chains failed
        theta_r1_NLE = theta_r1_NLE[250:]
    if task_id == 9:
        theta_r1_NLE = theta_r1_NLE[:750]

    pred_ys_NLE = psi_grids @ torch.linalg.inv(A) @ theta_r1_NLE.T

    # NPE res
    theta_r1_NPE = torch.tensor(np.load(f"res_NPE_newdeepsets/theta_post{task_id}.npy"), dtype=torch.float32)
    pred_ys_NPE = psi_grids @ torch.linalg.inv(A) @ theta_r1_NPE.T

    # ====== Get the weight for ABCW1 and NPE: w(\theta) \propto \frac{1}{q(\theta)}, where q(\theta) is the proposal density ====== #
    theta_pre = torch.tensor(pd.read_csv(f"res_SW1_precond/theta_pre_task{task_id}.csv").values, dtype = torch.float32).contiguous()
    
    lower = torch.zeros(M + 1) # .to(device)
    lower[0] = a0
    lower[1:] = a
    upper = torch.zeros(M + 1) # .to(device)
    upper[0] = b0
    upper[1:] = b
    
    actual_inf_rate = torch.ones(M + 1)
    actual_inf_rate[-2:] = 2
    
    inf_rate = torch.zeros(M + 1)
    for i in range(M + 1):
        inf_rate[i] = get_inf_rate(mode = theta_pre.mean(dim = 0)[i].item(), std_orig = theta_pre.std(dim = 0)[i].item(),
                        lower = lower[i].item(), upper = upper[i].item(), actual_inf_rate = actual_inf_rate[i].item())
    
    
    mean_theta = theta_pre.mean(dim = 0)
    std_theta = inf_rate * theta_pre.std(dim = 0)
    
    # the proposal distribution
    dist = MultivariateNormal(loc = mean_theta, covariance_matrix = torch.diag(std_theta**2))
    
    ### Weight for NPE
    log_weight_NPE = -dist.log_prob(theta_r1_NPE)
    weight_NPE = normalized_weight(log_weight_NPE)
    
    ### Weight for ABCW1
    log_weight_ABCW1 = -dist.log_prob(theta_r1_ABCW1)
    weight_ABCW1 = normalized_weight(log_weight_ABCW1)

    
    A = get_A(M)
    grids = torch.arange(0, 1.01, 0.01)
    psi_grids = get_psi(grids, M)


    ######## Calculation begins
    
    # KS error
    for i in range(pred_ys_gibbs.shape[0]):
        KS_fisherDR_cross[j, i] = ks_2samp(pred_ys_gibbs[i], pred_ys_fisherDR_cross[i]).statistic
        KS_nmodel_fisher[j, i] = ks_2samp(pred_ys_gibbs[i], pred_ys_nmodel_fisher[i]).statistic
        KS_nmodel_fisher_5x[j, i] = ks_2samp(pred_ys_gibbs[i], pred_ys_nmodel_fisher_5x[i]).statistic
        KS_ABCW1[j, i] = weighted_ks_2samp(pred_ys_gibbs[i], pred_ys_ABCW1[i], wx=None, wy=weight_ABCW1)
        KS_NPE[j, i] = weighted_ks_2samp(pred_ys_gibbs[i], pred_ys_NPE[i], wx=None, wy=weight_NPE)
        KS_BSL[j, i] = ks_2samp(pred_ys_gibbs[i], pred_ys_BSL[i]).statistic
        KS_NLE[j, i] = ks_2samp(pred_ys_gibbs[i], pred_ys_NLE[i]).statistic

    # W1 error
    for i in range(pred_ys_gibbs.shape[0]):
        W1_fisherDR_cross[j, i] = ot.wasserstein_1d(pred_ys_gibbs[i], pred_ys_fisherDR_cross[i])
        W1_nmodel_fisher[j, i] = ot.wasserstein_1d(pred_ys_gibbs[i], pred_ys_nmodel_fisher[i])
        W1_nmodel_fisher_5x[j, i] = ot.wasserstein_1d(pred_ys_gibbs[i], pred_ys_nmodel_fisher_5x[i])
        W1_ABCW1[j, i] = ot.wasserstein_1d(u_values = pred_ys_gibbs[i], v_values = pred_ys_ABCW1[i],
                                          u_weights = None, v_weights = weight_ABCW1) 
        W1_NPE[j, i] = ot.wasserstein_1d(u_values = pred_ys_gibbs[i], v_values = pred_ys_NPE[i],
                                          u_weights = None, v_weights = weight_NPE) 
        W1_BSL[j, i] = ot.wasserstein_1d(pred_ys_gibbs[i], pred_ys_BSL[i])
        W1_NLE[j, i] = ot.wasserstein_1d(pred_ys_gibbs[i], pred_ys_NLE[i])
    
    # CI width
    for i in range(pred_ys_gibbs.shape[0]):
        Width_fisherDR_cross[j, i] = torch.quantile(pred_ys_fisherDR_cross[i], 0.975) - torch.quantile(pred_ys_fisherDR_cross[i], 0.025)
        Width_nmodel_fisher[j, i] = torch.quantile(pred_ys_nmodel_fisher[i], 0.975) - torch.quantile(pred_ys_nmodel_fisher[i], 0.025)
        Width_nmodel_fisher_5x[j, i] = torch.quantile(pred_ys_nmodel_fisher_5x[i], 0.975) - torch.quantile(pred_ys_nmodel_fisher_5x[i], 0.025)
        Width_ABCW1[j, i] = weighted_quantile(pred_ys_ABCW1[i], weight_ABCW1, 0.975) - weighted_quantile(pred_ys_ABCW1[i], weight_ABCW1, 0.025)
        Width_NPE[j, i] = weighted_quantile(pred_ys_NPE[i], weight_NPE, 0.975) - weighted_quantile(pred_ys_NPE[i], weight_NPE, 0.025)
        Width_BSL[j, i] = torch.quantile(pred_ys_BSL[i], 0.975) - torch.quantile(pred_ys_BSL[i], 0.025)
        Width_NLE[j, i] = torch.quantile(pred_ys_NLE[i], 0.975) - torch.quantile(pred_ys_NLE[i], 0.025)

        Width_gibbs[j, i] = torch.quantile(pred_ys_gibbs[i], 0.975) - torch.quantile(pred_ys_gibbs[i], 0.025)
        
    # Coverage
    true_mean = torch.tanh(4.0 * grids - 2.0)
    for i in range(pred_ys_gibbs.shape[0]):
        Cvg_fisherDR_cross[j, i] = 1.0 * ( true_mean[i] > torch.quantile(pred_ys_fisherDR_cross[i], 0.025) and true_mean[i] < torch.quantile(pred_ys_fisherDR_cross[i], 0.975) )
        Cvg_nmodel_fisher[j, i] = 1.0 * ( true_mean[i] > torch.quantile(pred_ys_nmodel_fisher[i], 0.025) and true_mean[i] < torch.quantile(pred_ys_nmodel_fisher[i], 0.975) )
        Cvg_nmodel_fisher_5x[j, i] = 1.0 * ( true_mean[i] > torch.quantile(pred_ys_nmodel_fisher_5x[i], 0.025) and true_mean[i] < torch.quantile(pred_ys_nmodel_fisher_5x[i], 0.975) )
        Cvg_ABCW1[j, i] = 1.0 * ( true_mean[i] > weighted_quantile(pred_ys_ABCW1[i], weight_ABCW1, 0.025) and true_mean[i] < weighted_quantile(pred_ys_ABCW1[i], weight_ABCW1, 0.975) )
        Cvg_NPE[j, i] = 1.0 * ( true_mean[i] > weighted_quantile(pred_ys_NPE[i], weight_NPE, 0.025) and true_mean[i] < weighted_quantile(pred_ys_NPE[i], weight_NPE, 0.975) )
        Cvg_BSL[j, i] = 1.0 * ( true_mean[i] > torch.quantile(pred_ys_BSL[i], 0.025) and true_mean[i] < torch.quantile(pred_ys_BSL[i], 0.975) )
        Cvg_NLE[j, i] = 1.0 * ( true_mean[i] > torch.quantile(pred_ys_NLE[i], 0.025) and true_mean[i] < torch.quantile(pred_ys_NLE[i], 0.975) )
        
        Cvg_gibbs[j, i] = 1.0 * ( true_mean[i] > torch.quantile(pred_ys_gibbs[i], 0.025) and true_mean[i] < torch.quantile(pred_ys_gibbs[i], 0.975) )

In [ ]:
res = {'single-model': [KS_fisherDR_cross.mean().item(), W1_fisherDR_cross.mean().item(), Cvg_fisherDR_cross.mean().item(), Width_fisherDR_cross.mean().item()],
      'n-model': [KS_nmodel_fisher.mean().item(), W1_nmodel_fisher.mean().item(), Cvg_nmodel_fisher.mean().item(), Width_nmodel_fisher.mean().item()],
       'n-model-5x': [KS_nmodel_fisher_5x.mean().item(), W1_nmodel_fisher_5x.mean().item(), Cvg_nmodel_fisher_5x.mean().item(), Width_nmodel_fisher_5x.mean().item()],
       'ABC-W1': [KS_ABCW1.mean().item(), W1_ABCW1.mean().item(), Cvg_ABCW1.mean().item(), Width_ABCW1.mean().item()],
       'BSL': [KS_BSL.mean().item(), W1_BSL.mean().item(), Cvg_BSL.mean().item(), Width_BSL.mean().item()],
       'NPE': [KS_NPE.mean().item(), W1_NPE.mean().item(), Cvg_NPE.mean().item(), Width_NPE.mean().item()],
       'NLE': [KS_NLE.mean().item(), W1_NLE.mean().item(), Cvg_NLE.mean().item(), Width_NLE.mean().item()],
      'Truth': ["-", "-", Cvg_gibbs.mean().item(), Width_gibbs.mean().item()]}
index_names = ['KS', 'W1', 'Coverage', 'Width']
df = pd.DataFrame(res, index=index_names)
df.T

In [ ]:
def format_general(x):
    """Format all columns except W1 to 3 decimal places."""
    return f"{x:.3f}" if isinstance(x, (int, float)) else x

def format_w1(x):
    """Format W1 column with 3 decimal places scaled by 10^-2."""
    return f"{x * 100:.3f}" if isinstance(x, (int, float)) else x

# Apply formatting only to numeric columns (excluding W1 separately)
df_formatted = df.T.apply(lambda col: col.map(format_w1) if col.name == "W1" else col.map(format_general))

In [ ]:
df_formatted.columns = ["KS", "W1 ($\times 10^{-2}$)", "Cover", "CI width"]
df_formatted

In [ ]:
latex_code = df_formatted.to_latex(index=True, float_format="%.3f", escape=False)
print(latex_code)